# Chapter 2 · Populations, Samples, and Data: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, the January register of discharges to home of a 500-bed hospital, through the manager's questions; only the sections with a calculation appear (2.7, 2.9, and 2.10). **Part B** gives the written solution of every chapter exercise (1 to 9; statements are in the book) and, for the exercises whose solution needs a calculation (7, 8, and 9), its code and result. The code is plain Python 3 with its standard library only. Run the cells in order.

## Part A. The case

**Data.** `data/register.csv`: 1,200 rows, one per discharge episode of January 2026, and five columns: `episode_id`, `patient_id`, `formalized_at` (when the discharge was formalized), `departed_at` (when the patient left), and `delayed` (already computed from the two times; the first cell computes it again). Each cell carries the number of the book section it belongs to.

### 2.7 Defining the event to be counted

*When does a discharge count as delayed, and how many of the 120 sampled episodes were?*

In [1]:
# AUDIT PROGRAM: how long did the sampled patients wait after the
# discharge was formalized, and how many waited more than 2 hours?
#
# Pseudocode
#   1. Read the register of January discharges, one record per episode.
#   2. For each episode, compute the wait in hours, from the formalized
#      discharge to the departure. A missing time leaves it unknown.
#   3. Mark the episode as delayed if the wait is MORE than 2 hours.
#   4. Draw the audit sample: 120 episodes, at random, without replacement.
#   5. Report on the sample: delayed, percentage, median, longest, unknown.
#   6. Count again under the rule "2 hours or more", to see what
#      changes when a wait of exactly 2 hours counts as delayed.

import csv
import random
import statistics
from datetime import datetime

TARGET_HOURS = 2  # the audit's rule: delayed means MORE than 2 hours

# 1. Read the register.
with open("data/register.csv", newline="") as stream:
    population = list(csv.DictReader(stream))

# 2-3. Compute each wait and mark the delays.
for row in population:
    row["wait"] = None
    row["delayed"] = None
    if row["formalized_at"] and row["departed_at"]:
        formalized = datetime.fromisoformat(row["formalized_at"])
        departed = datetime.fromisoformat(row["departed_at"])
        wait = (departed - formalized).total_seconds() / 3600
        if wait >= 0:
            row["wait"] = wait
            row["delayed"] = wait > TARGET_HOURS
        else:
            print("Check timestamps:", row["episode_id"])

# 4. Draw the sample; the seed (11) makes the draw reproducible.
sample = random.Random(11).sample(population, 120)

# 5. Report on the sample.
waits = [row["wait"] for row in sample if row["wait"] is not None]
delayed = [wait for wait in waits if wait > TARGET_HOURS]
print("Episodes reviewed:", len(sample))
print("Wait unknown:", len(sample) - len(waits))
print("Over 2 hours:", len(delayed), "of", len(waits))
print("Percentage over 2 hours:", 100 * len(delayed) / len(waits))
print("Median wait (hours):", statistics.median(waits))
print("Longest wait (hours):", max(waits))

# 6. The same waits under the rule "2 hours or more".
at_least_two = [wait for wait in waits if wait >= TARGET_HOURS]
share = 100 * len(at_least_two) / len(waits)
print("2 hours or more:", len(at_least_two), "of", len(waits))
print("Percentage 2 hours or more:", share)

Episodes reviewed: 120
Wait unknown: 0
Over 2 hours: 24 of 120
Percentage over 2 hours: 20.0
Median wait (hours): 1.5
Longest wait (hours): 8.0
2 hours or more: 45 of 120
Percentage 2 hours or more: 37.5


### 2.9 Sampling variation

*Would another random draw of 120 episodes give the same proportion?*

In [2]:
# SAMPLING VARIATION: do different random draws give the same proportion?
#
# Pseudocode
#   1. Count the delayed episodes in the whole register (possible only
#      because the data are synthetic; an audit team would not have it).
#   2. Draw three samples of 120 episodes, each with a different seed.
#   3. For each sample, count the delayed episodes and the percentage.

def count_delayed(records):
    """Number of records whose delay status is True (unknown is not counted)."""
    count = 0
    for row in records:
        if row["delayed"] is True:
            count = count + 1
    return count

# 1. The whole register.
N = len(population)
K = count_delayed(population)
print(f"Whole register: {K} delayed of {N} ({100 * K / N:.2f}%)")

# 2-3. Three samples; the seeds 11, 17, and 42 make the draws reproducible.
sample_number = 0
for seed in [11, 17, 42]:
    sample_number = sample_number + 1
    draw = random.Random(seed).sample(population, 120)
    k = count_delayed(draw)
    n = len(draw)
    print(f"Sample {sample_number}: {k} delayed of {n} ({100 * k / n:.2f}%)")

Whole register: 216 delayed of 1200 (18.00%)
Sample 1: 24 delayed of 120 (20.00%)
Sample 2: 22 delayed of 120 (18.33%)
Sample 3: 18 delayed of 120 (15.00%)


### 2.10 Missing observations

*What can the team report when the time of the formalized discharge is missing for 12 of the 120 records?*

In [3]:
# MISSING OBSERVATIONS: what can the team report when times are missing?
#
# Pseudocode
#   1. Make a separate copy of the 120 sampled records.
#   2. In the copy, hide the delay status of the first 12 episodes, as if
#      their formalized discharge time had not been recorded.
#   3. Count the selected, known, and unknown statuses and the known delays.
#   4. Report the proportion delayed among the known statuses.
#   5. Report the range for all 120 selected episodes: every unknown status
#      timely (lowest) or every unknown status delayed (highest).

# 1-2. Copy the records (the originals stay intact) and hide 12 statuses.
incomplete = [dict(row) for row in sample]
for row in incomplete[:12]:
    row["delayed"] = None

# 3. Counts.
selected = len(incomplete)
known = [row for row in incomplete if row["delayed"] is not None]
unknown = selected - len(known)
known_delays = count_delayed(known)

# 4-5. Results.
print("Selected:", selected)
print("Known status:", len(known))
print("Unknown status:", unknown)
print("Known delays:", known_delays)
print(f"Delayed among known: {100 * known_delays / len(known):.1f}%")
lowest = 100 * known_delays / selected
highest = 100 * (known_delays + unknown) / selected
print(f"All selected: {lowest:.1f}% to {highest:.1f}%")

Selected: 120
Known status: 108
Unknown status: 12
Known delays: 23
Delayed among known: 21.3%
All selected: 19.2% to 29.2%


## Part B. Exercise solutions

### 2.13 Exercise 1. Beds, episodes, and patients

The unit is one discharge to home, the home discharge episode. The target population
consists of the 1,200 eligible discharge episodes at this hospital during January.
The 500 beds describe capacity; 1,080 is the number of distinct patients associated
with those episodes. Neither is the number of episodes.

A patient can contribute two eligible episodes. Both belong in an episode-based
audit. If the question instead concerns patients with at least one delayed discharge,
the unit becomes the patient, and the target population consists of the 1,080 distinct
patients associated with the eligible episodes. Their stays must be linked so that
each person is counted once and a delay in any eligible stay is identified. Counting
episodes cannot by itself provide the patient-level proportion.

### 2.13 Exercise 2. Identify the relevant interval

The interval from formalized discharge to departure is 2.5 hours, exceeding the two-hour reference.
The additional 30 minutes before the bed becomes available concern preparation of
the bed, not the patient's waiting after the discharge was formalized. The third timestamp would allow
a separate investigation of when vacated beds become available for new admissions.

### 2.13 Exercise 3. An incomplete sampling frame

The target population contains all 1,200 eligible episodes. The frame contains only
the 900 weekday episodes. The sample contains the 300 selected episodes. The 300
weekend episodes have no chance of selection from this frame. Selecting 600 rather
than 300 weekday episodes would still exclude every weekend episode.

Request a corrected export with all eligible days and one entry per episode, reconcile
it to the 1,200-episode register, and then select according to the documented plan.
The frame's coverage must be corrected before a larger sample can address the
original question.

### 2.13 Exercise 4. A census has a defined boundary

If all 240 eligible appointments are included, the coverage is a census of April
appointments. A person with three scheduled appointments contributes three units to
that census. A patient-level question requires a separate definition and linkage of
appointments belonging to the same person. The appointment list also need not include
every patient registered with the clinic.

Complete appointment coverage does not verify the accuracy of the attendance entries.
It also does not establish May's attendance pattern, when the scheduled appointments,
patients, and service conditions may differ. Coverage, recording accuracy, and
application to another period are separate questions.

### 2.13 Exercise 5. A parameter and a statistic for rejected specimens

The population parameter is the proportion rejected among all 800 eligible specimens:

$$ p=\frac{40}{800}=0.05=5\%. $$

The sample statistic is the proportion rejected among the 80 selected specimens:

$$ \widehat{p}=\frac{6}{80}=0.075=7.5\%. $$

The sample statistic is 2.5 percentage points higher. The sample contains only a
selection of specimens; another selection can contain a different number rejected.
The population still contains 800 specimens and 40 rejections. No uncertainty interval
or sample-size recommendation follows from this comparison alone.

### 2.13 Exercise 6. Records, duplicates, and missing observations

The unit is one rehabilitation visit. The entries identify two patients, P01 and P02, and three distinct visits, V01, V02,
and V03. P01's two visits are legitimate separate units. V03 appears twice with the
same recorded information. Check the source to confirm whether this is a duplicated
entry before retaining a single record for that visit.

V03's duration is unknown. Zero minutes would assert a known duration and replace
missing information with a different claim. The service has evidence of three
visits, but duration information for only two of them. Those counts should remain
separate.

### 2.13 Exercise 7. Three draws from one target population

The code of this and the next exercise is in Part B of the companion notebook, which
first runs the chapter's cells through Section 2.9: they read the register,
classify delays, and create the samples used below.

Samples 1, 2, and 3 contain 24, 22, and 18 delayed episodes,
respectively: 20.00%, 18.33%, and 15.00%. The target population remains 1,200 episodes,
including 216 delays (18%). Each draw selects 120 distinct entries without replacement.

To check uniqueness, the notebook collects the episode identifiers into a `set`. A set
keeps each distinct value once; comparing its size with the sample size detects
repeated identifiers. An `assert` statement checks a condition and stops
execution if the condition is false; here it checks the printed numerical claims.

All checks pass: each sample contains 120 distinct episodes, and the register
still contains 1,200 episodes and 216 delays. The samples contain different
episodes, so their proportions can differ. Three
realized samples illustrate variation; they do not characterize its full distribution.

In [4]:
for seed in [11, 17, 42]:
    draw = random.Random(seed).sample(population, 120)
    identifiers = set()
    for row in draw:
        identifiers.add(row["episode_id"])
    assert len(draw) == len(identifiers) == 120
assert len(population) == 1200
assert K == 216

### 2.13 Exercise 8. An incomplete sample

The sample contains 120 selected episodes: 108 known statuses, 12 unknown statuses,
and 23 known delays. Among known statuses, $23/108\approx21.3\%$. Among all selected
episodes, the number delayed can be as small as 23 or as large as $23+12=35$,
giving approximately 19.2%-29.2%. The latter range describes the selected group
under missing information; it is not a population confidence interval.

The notebook recreates Sample 1, hides 12 statuses in a separate copy,
and confirms the observed proportion and the two bounds calculated above.

The notebook's output confirms 23 known delays among 108 observed statuses (21.3%).
The bounds use all 120 selected episodes: assigning the 12 unknown statuses
to timely or delayed gives 19.2% or 29.2%, respectively.

In [5]:
sample = random.Random(11).sample(population, 120)
incomplete = []
for row in sample:
    incomplete.append(dict(row))
for row in incomplete[:12]:
    row["delayed"] = None

known_n = 0
known_delays = 0
for row in incomplete:
    if row["delayed"] is not None:
        known_n = known_n + 1
        if row["delayed"] is True:
            known_delays = known_delays + 1
selected_n = len(incomplete)
missing_n = selected_n - known_n
print(selected_n, known_n, missing_n, known_delays)
# 120 108 12 23
print(f"Known statuses: {100 * known_delays / known_n:.1f}%")
# Known statuses: 21.3%
lower = 100 * known_delays / selected_n
upper = 100 * (known_delays + missing_n) / selected_n
print(f"Selected group: {lower:.1f}%-{upper:.1f}%")
# Selected group: 19.2%-29.2%

120 108 12 23
Known statuses: 21.3%
Selected group: 19.2%-29.2%


### 2.13 Exercise 9. Define the next process audit

The post-vacating intervals total $20+50+30+10+40=150$ minutes. Waiting for cleaning
to start is longest at 50 minutes; cleaning itself takes 30 minutes.

For a January study of that wait, one bed-vacating event is the unit. Define the
eligible wards and dates, list all eligible vacating events, and obtain the vacating,
request, and cleaning-start timestamps. Include events still awaiting cleaning;
restricting the register to completed cleanings would exclude the unresolved waits.
The team must also define the observation cutoff and record each event's status at
that time. Review assignment and staffing records to investigate why a wait occurred.
One illustrated event identifies an interval to study, not its hospital-wide frequency.

In [6]:
# 1. The six events of one synthetic bed-vacating episode (from Figure 2.2)
events = [
    ("vacated", "2026-01-04T13:00"),
    ("cleaning requested", "2026-01-04T13:20"),
    ("cleaning started", "2026-01-04T14:10"),
    ("cleaning finished", "2026-01-04T14:40"),
    ("bed ready", "2026-01-04T14:50"),
    ("next patient in", "2026-01-04T15:30"),
]

# 2. Minutes between each event and the next, with the longest interval
total = 0
longest_name = ""
longest_minutes = 0
for i in range(len(events) - 1):
    name_a, time_a = events[i]
    name_b, time_b = events[i + 1]
    minutes = (datetime.fromisoformat(time_b) - datetime.fromisoformat(time_a)).total_seconds() / 60
    print(name_a, "to", name_b + ":", minutes, "minutes")
    total = total + minutes
    if minutes > longest_minutes:
        longest_minutes = minutes
        longest_name = name_a + " to " + name_b
print("Total from bed vacating to the next patient:", total, "minutes")
print("Longest interval:", longest_name, "(" + str(longest_minutes) + " minutes)")

vacated to cleaning requested: 20.0 minutes
cleaning requested to cleaning started: 50.0 minutes
cleaning started to cleaning finished: 30.0 minutes
cleaning finished to bed ready: 10.0 minutes
bed ready to next patient in: 40.0 minutes
Total from bed vacating to the next patient: 150.0 minutes
Longest interval: cleaning requested to cleaning started (50.0 minutes)


---
*Synthetic data (`generate_data.py`, no random numbers). Teaching values only; nothing here is an estimate for a real hospital.*